# Control-store baseline: BSTS with the control store as an input

Click **Run all** (or run the cells from top to bottom). All the code is in this notebook; the only file you need is `Walmart_Sales.csv`.

* **Colab:** the notebook mounts Google Drive and saves its progress in `MyDrive/bsts_run`. Put `Walmart_Sales.csv` in that folder. **Elsewhere:** it works in the current folder.
* If the notebook stops (for example Colab disconnects), run it again from the top. Finished fits are saved and are not repeated.
* If `Walmart_Sales.csv` cannot be found, the notebook stops with an error that lists where it looked.
* In the cell *Settings and run*, start with `RUN = "pilot"` (quick test), then change it to `"full"` for the real run.
* It needs a Python version that TensorFlow supports (Colab's default works).

## 1. Working folder (Google Drive in Colab)

In [1]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # quieter TensorFlow
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/bsts_run', exist_ok=True)
    os.chdir('/content/drive/MyDrive/bsts_run')
    print("Running in Colab: progress is saved in Google Drive.")
except ImportError:
    print("Not running in Colab: using the current folder; progress is saved here.")
print("Working folder:", os.getcwd())
print("Files here:", os.listdir('.'))

Not running in Colab: using the current folder; progress is saved here.
Working folder: c:\Users\user\AppData\Local\Programs\Microsoft VS Code
Files here: ['bin', 'Code.exe', 'Code.VisualElementsManifest.xml', 'e646868227', 'unins000.dat', 'unins000.exe', 'unins000.msg']


## 2. Install the BSTS package

In [2]:
import sys, os, subprocess, importlib, importlib.metadata as md

def _pip(*args):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                          capture_output=True, text=True)

def _have_causalimpact():
    importlib.invalidate_caches()
    try:
        from causalimpact import CausalImpact  # noqa: F401
        return True
    except Exception as exc:
        print("causalimpact is not importable yet:", type(exc).__name__, str(exc)[:200])
        return False

if not _have_causalimpact():
    r = _pip("tfcausalimpact")
    if r.returncode != 0:            # newer setuptools can reject this package's metadata
        print("Plain install failed; retrying with an older setuptools.")
        _pip("setuptools<70", "wheel")
        r = _pip("--no-build-isolation", "tfcausalimpact")
    if r.returncode != 0:
        print(r.stderr[-1500:])
    if not _have_causalimpact():     # last try: legacy Keras for TensorFlow Probability
        _pip("tf-keras")
        os.environ["TF_USE_LEGACY_KERAS"] = "1"
    if not _have_causalimpact():
        raise RuntimeError("tfcausalimpact could not be installed or imported on Python "
                           + sys.version.split()[0] + ". Use Colab, or a Python version "
                           "that TensorFlow supports, and run again.")
try:
    print("tfcausalimpact version:", md.version("tfcausalimpact"))
except md.PackageNotFoundError:
    print("causalimpact is importable (package version not reported).")

g++ not available, if using conda: `conda install gxx`


causalimpact is importable (package version not reported).


## 3. Configuration (do not edit)

In [3]:
# ============================================================
# STEP 3: ONE SCRIPT FOR ALL PAPER TABLES
# ============================================================
# Produces every number reported in Section 8:
#   coverage, interval width, estimation error (MAE, bias),
#   shift-detection rate, false-alarm rate, point-estimate correction,
#   calibration curves, and the 20-split stability check.
#
# Protocol (paper Section 7.8):
#   * 45 stores are split at random into two disjoint groups
#     (22 calibration stores / 23 test stores). Only pairs with BOTH
#     stores in the same group are used, so no store is in both sets.
#   * lambda (break-aware) and k (fixed widening) are chosen on the
#     CALIBRATION pairs only: the smallest grid value with >= 95%
#     coverage in BOTH the no-shift and the shift scenario.
#   * The detection threshold c is chosen on the calibration no-shift
#     control stores with a finite-sample (conformal) rule, which holds the
#     false-alarm rate at 5% or less on average. The plain 95% quantile of
#     only 22 stores undershoots and gives about 10%.
#   * The control diagnostic is the year-over-year log-ratio change
#     (DIAGNOSTIC = "yoy"). The earlier trend + sine diagnostic is kept as
#     DIAGNOSTIC = "legacy" for comparison; it cannot separate shifted from
#     unshifted controls (AUC 0.34, worse than chance).
#   * Everything reported in the tables comes from the TEST pairs.
#
# ENGINE = "seasonal_naive": quick benchmark (no BSTS needed).
# ENGINE = "bsts":           standard (no-covariate) BSTS CausalImpact.
#                            One fit per treated store and scenario
#                            (45 x 2 = 90 fits), cached on disk.
# ENGINE, CONTROL_BASELINE and MAX_CONTROL_PAIRS are set by the
# "Settings and run" cell at the end of the notebook (RUN = ...).
# ============================================================

import json
import os
import sys
import warnings
from itertools import permutations
from pathlib import Path

import logging
import numpy as np
import pandas as pd

logging.getLogger("tensorflow").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------ CONFIG ------------------------------
ENGINE = "bsts"                    # set by the run cell
BSTS_SEED = 0                      # makes the BSTS fits repeatable
# Control-store baseline: BSTS CausalImpact with the control store as an
# additional input (covariate), fitted separately for every pair.
#   "off":     not run (default)
#   "test":    the 506 test pairs of the primary split x 2 scenarios
#   "primary": the 462 calibration pairs + the 506 test pairs of the primary
#              split x 2 scenarios (1,936 fits). The calibration pairs are
#              used to choose a widening k for the control-store method.
#   "all":     all 1,980 pairs x 2 scenarios (needed for the 20-split summary)
CONTROL_BASELINE = "primary"       # set by the run cell
MAX_CONTROL_PAIRS = 20             # set by the run cell (None = all pairs)

CSV_CANDIDATES = ["C:\\Users\\user\\Downloads\\Walmart_Sales.csv"]

OUTPUT_DIR = Path("paper_tables")

PRE_WEEKS = 100
TREATMENT_EFFECT = 0.03            # injected into the treated store
BUSINESS_SHIFT = 0.05              # common shift, shift scenario only
# How the 3% effect and the 5% shift combine in the treated store:
#   "additive":       post-period sales x (1 + 0.03 + 0.05)  (reproduces
#                     the paper's tables)
#   "multiplicative": post-period sales x 1.03 x 1.05       (what the
#                     earlier notebook cells 0 and 2 do)
SHIFT_COMPOSITION = "additive"
Z95 = 1.96                         # as in the paper's formulas
HAC_LAG = 4                        # Newey-West lags (Bartlett weights)
NOMINAL = 0.95

# Control diagnostic used in the break-aware interval, the detection
# tables and the point-estimate correction:
#   "yoy":    year-over-year log-ratio level change (holiday-robust)
#   "legacy": trend + annual sine fitted on the pre-period (the diagnostic
#             in the earlier draft; kept for comparison)
DIAGNOSTIC = "yoy"
SHIFT_SIZES = [0.02, 0.03, 0.05, 0.08, 0.10]   # detector power curve

N_CAL_STORES = 22                  # 22 calibration stores -> 462 pairs
N_SPLITS = 20                      # primary split = split 0
SPLIT_SEED = 0
N_BOOTSTRAP = 2000

# Parameter grids. The paper says only "a fixed grid"; these grids
# reproduce its primary-split selections (lambda=100, k=8). Changing them
# can change the 20-split averages slightly.
LAMBDA_GRID = [0, 0.25, 0.5, 1, 2, 5, 10, 15, 20, 30, 50, 75, 100,
               150, 200, 300, 500]
K_GRID = [1, 1.5, 2, 3, 4, 5, 6, 8, 10, 12, 16, 20, 30, 50, 100]
LAMBDA_DEMO = 0.25                 # for the Store 1 / Store 2 example
DEMO_TREATED, DEMO_CONTROL = 1, 2

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCENARIOS = ("noshift", "shift")

## 4. Check the data file (stops with an error if it is missing)

In [4]:
from pathlib import Path
_found = [p for p in CSV_CANDIDATES if Path(p).exists()]
if not _found:
    raise FileNotFoundError(
        "Walmart_Sales.csv is NOT available. Looked in: " + ", ".join(CSV_CANDIDATES)
        + f" (working folder: {os.getcwd()}). Put the file in the working folder and run again.")
_cols = list(pd.read_csv(_found[0], nrows=0).columns)
_missing = [c for c in ("Store", "Date", "Weekly_Sales") if c not in _cols]
if _missing:
    raise ValueError(f"{_found[0]} is missing the columns {_missing}; it has {_cols}.")
print("OK: Walmart_Sales.csv found at", _found[0])

OK: Walmart_Sales.csv found at C:\Users\user\Downloads\Walmart_Sales.csv


## 5. Data and scenarios (do not edit)

In [5]:
# ------------------------------ DATA ------------------------------
def load_walmart():
    path = next((Path(p) for p in CSV_CANDIDATES if Path(p).exists()), None)
    if path is None:
        raise FileNotFoundError(
            "Walmart_Sales.csv not found. Looked in: " + ", ".join(CSV_CANDIDATES)
            + f" (working folder: {os.getcwd()}). Put the file there and run again.")
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
    df = df.dropna(subset=["Date", "Store", "Weekly_Sales"])
    wide = (df.pivot_table(index="Date", columns="Store",
                           values="Weekly_Sales", aggfunc="mean")
              .sort_index().dropna(axis=1))
    if len(wide) <= PRE_WEEKS:
        raise ValueError("Not enough weeks.")
    return wide


# ------------------------------ SCENARIOS ------------------------------
def make_series(orig, scenario, is_treated):
    """Semi-synthetic series. The 3% effect goes into the treated store.
    The 5% shift goes into BOTH stores after the intervention. The true
    effect is the injected 3% term only (paper Section 7.5)."""
    y = orig.astype(float).copy()
    post = np.arange(len(y)) >= PRE_WEEKS
    effect = TREATMENT_EFFECT if is_treated else 0.0
    shift = BUSINESS_SHIFT if scenario == "shift" else 0.0
    if SHIFT_COMPOSITION == "additive":
        y[post] *= 1 + effect + shift
    elif SHIFT_COMPOSITION == "multiplicative":
        y[post] *= (1 + effect) * (1 + shift)
    else:
        raise ValueError("SHIFT_COMPOSITION must be additive or multiplicative")
    return y


def true_effect(orig_treated):
    return float(TREATMENT_EFFECT * orig_treated[PRE_WEEKS:].sum())

## 6. Control diagnostics and BSTS intervals (do not edit)

In [6]:
# ------------------------------ HAC / DIAGNOSTIC ------------------------------
def nw_long_run_variance(e, lags=HAC_LAG):
    e = np.asarray(e, float)
    e = e - e.mean()
    n = len(e)
    s = e @ e / n
    for l in range(1, lags + 1):
        s += 2 * (1 - l / (lags + 1)) * (e[l:] @ e[:-l] / n)
    return float(max(s, 0.0))


def _design(t):
    t = np.asarray(t, float)
    return np.column_stack([np.ones(len(t)), t,
                            np.sin(2 * np.pi * t / 52),
                            np.cos(2 * np.pi * t / 52)])


def legacy_diagnostic(control):
    """Earlier diagnostic: trend + annual sine fitted on the pre-period;
    z = mean post-period residual / its HAC standard error."""
    t = np.arange(len(control))
    beta, *_ = np.linalg.lstsq(_design(t[:PRE_WEEKS]),
                               control[:PRE_WEEKS], rcond=None)
    resid = control - _design(t) @ beta
    post = resid[PRE_WEEKS:]
    delta = float(post.mean())
    se = float(np.sqrt(nw_long_run_variance(post) / len(post)))
    return {"z": delta / se if se > 0 else 0.0, "delta": delta,
            "hac_se": se, "pre_mean": float(control[:PRE_WEEKS].mean())}


def yoy_diagnostic(control):
    """Year-over-year diagnostic. For each week t >= 52 the log ratio
    log N_t - log N_(t-52) removes trend, annual seasonality and holidays
    by construction. The statistic S is the mean post-period log ratio
    minus the mean pre-period log ratio (a change in year-over-year
    growth); z = S / HAC standard error of the difference. A common
    multiplicative shift of size s moves S by about log(1 + s)."""
    lr = np.log(control[52:]) - np.log(control[:-52])
    pre, post = lr[:PRE_WEEKS - 52], lr[PRE_WEEKS - 52:]
    S = float(post.mean() - pre.mean())
    se = float(np.sqrt(nw_long_run_variance(post) / len(post)
                       + nw_long_run_variance(pre) / len(pre)))
    return {"z": S / se if se > 0 else 0.0, "S": S,
            "rel": float(np.expm1(S)), "hac_se": se}


def control_diagnostics(control):
    return {"legacy": legacy_diagnostic(control),
            "yoy": yoy_diagnostic(control)}


# ------------------------------ STANDARD INTERVAL ------------------------------
def seasonal_naive_interval(y):
    """Forecast each post week with the same week 52 weeks earlier. The
    standard error of the cumulative effect comes from the pre-period
    seasonal-naive errors (Newey-West long-run variance)."""
    post = np.arange(PRE_WEEKS, len(y))
    est = float((y[post] - y[post - 52]).sum())
    pre = np.arange(52, PRE_WEEKS)
    sigma = float(np.sqrt(len(post) *
                          nw_long_run_variance(y[pre] - y[pre - 52])))
    return est, sigma


def bsts_interval(y, dates, control=None):
    from causalimpact import CausalImpact
    # Fix the random seeds so every run gives the same BSTS numbers.
    np.random.seed(BSTS_SEED)
    try:
        import tensorflow as tf
        tf.random.set_seed(BSTS_SEED)
    except Exception:
        pass
    cols = {"treated_sales": y}
    if control is not None:           # control store as an additional input
        cols["control_sales"] = control
    data = pd.DataFrame(cols, index=dates)
    pre = [str(dates[0].date()), str(dates[PRE_WEEKS - 1].date())]
    post = [str(dates[PRE_WEEKS].date()), str(dates[-1].date())]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        impact = CausalImpact(data, pre, post)
    sm = impact.summary_data
    try:
        # Package versions differ: 'cumulative' is a row in some versions
        # and a column in others (e.g. tfcausalimpact 0.0.19).
        if "cumulative" in [str(c).strip().lower() for c in sm.columns]:
            col = [c for c in sm.columns if str(c).strip().lower() == "cumulative"][0]
            est = float(sm.loc["abs_effect", col])
            lo = float(sm.loc["abs_effect_lower", col])
            hi = float(sm.loc["abs_effect_upper", col])
        else:
            row = sm.loc[[i for i in sm.index
                          if str(i).strip().lower() == "cumulative"][0]]
            est = float(row["abs_effect"])
            lo, hi = float(row["abs_effect_lower"]), float(row["abs_effect_upper"])
    except Exception as exc:
        raise RuntimeError(
            "Could not read the cumulative effect from CausalImpact. "
            f"summary_data index: {list(sm.index)}, columns: {list(sm.columns)}"
        ) from exc
    return est, (hi - lo) / (2 * Z95)


def standard_interval(orig, scenario, dates, cache, key):
    if key in cache:
        return cache[key]
    y = make_series(orig, scenario, True)
    out = (seasonal_naive_interval(y) if ENGINE == "seasonal_naive"
           else bsts_interval(y, dates))
    cache[key] = out
    return out

## 7. Pair table and interval methods (do not edit)

In [7]:
# ------------------------------ BUILD PAIR TABLE ------------------------------
def build_pair_table(wide):
    dates = wide.index
    stores = list(wide.columns)
    cache_file = OUTPUT_DIR / f"cache_{ENGINE}.json"
    cache = {}
    if cache_file.exists():
        raw = json.loads(cache_file.read_text())
        cache = {tuple(k.split("|")): tuple(v) for k, v in raw.items()}
        cache = {(int(a), b): v for (a, b), v in cache.items()}

    std, ctrl = {}, {}
    for i, s in enumerate(stores, 1):
        orig = wide[s].to_numpy(float)
        for sc in SCENARIOS:
            if ENGINE == "bsts" and (s, sc) not in cache:
                print(f"  BSTS fit {i}/{len(stores)} store {s} [{sc}]")
            std[(s, sc)] = standard_interval(orig, sc, dates, cache, (s, sc))
            ctrl[(s, sc)] = control_diagnostics(make_series(orig, sc, False))
    if ENGINE == "bsts":
        cache_file.write_text(json.dumps(
            {f"{a}|{b}": list(v) for (a, b), v in cache.items()}))

    n_post = len(wide) - PRE_WEEKS
    rows = []
    for t_id, c_id in permutations(stores, 2):
        o_t = wide[t_id].to_numpy(float)
        truth = true_effect(o_t)
        y_pre = float(o_t[:PRE_WEEKS].mean())
        # treated post-period level without treatment: seasonal-naive
        # forecast (uses pre-period data only)
        base = float(o_t[PRE_WEEKS - 52:len(o_t) - 52].sum())
        for sc in SCENARIOS:
            est, sigma = std[(t_id, sc)]
            d = ctrl[(c_id, sc)]
            gamma = y_pre / d["legacy"]["pre_mean"]
            corr_legacy = gamma * d["legacy"]["delta"] * n_post
            corr_yoy = d["yoy"]["rel"] * base
            active = DIAGNOSTIC
            rows.append({
                "treated": t_id, "control": c_id, "scenario": sc,
                "est": est, "sigma": sigma, "truth": truth,
                "z": d[active]["z"],
                "z_legacy": d["legacy"]["z"], "z_yoy": d["yoy"]["z"],
                "delta": d["legacy"]["delta"], "hac_se": d["legacy"]["hac_se"],
                "yoy_S": d["yoy"]["S"], "yoy_rel": d["yoy"]["rel"],
                "corr_term": corr_yoy if active == "yoy" else corr_legacy,
            })
    return pd.DataFrame(rows)


# ------------------------------ INTERVAL METHODS ------------------------------
def half_standard(d):
    return Z95 * d["sigma"].to_numpy()

def half_fixed(d, k):
    return k * Z95 * d["sigma"].to_numpy()

def half_ba(d, lam):
    return (Z95 + lam * np.abs(d["z"].to_numpy())) * d["sigma"].to_numpy()

def covered(d, half):
    return np.abs(d["est"].to_numpy() - d["truth"].to_numpy()) <= half

def pct(x):
    return f"{100 * x:.2f}%"


def conformal_threshold(values, alpha=0.05):
    """Finite-sample threshold: the ceil((n+1)(1-alpha))-th smallest of n
    calibration values (n = distinct calibration CONTROL STORES, because
    pairs sharing a control share the same diagnostic). A new no-shift
    store exceeds it with probability <= alpha on average."""
    v = np.sort(np.asarray(values, float))
    k = int(np.ceil((len(v) + 1) * (1 - alpha)))
    return float(v[min(k, len(v)) - 1])


def select_param(cal, grid, half_fn):
    """Smallest grid value with >= 95% calibration coverage in BOTH
    scenarios. If none reaches it, the largest value is used and flagged."""
    curve = []
    chosen, reached = grid[-1], False
    for g in grid:
        covs = [covered(cal[cal.scenario == sc],
                        half_fn(cal[cal.scenario == sc], g)).mean()
                for sc in SCENARIOS]
        curve.append((g, *covs))
        if not reached and min(covs) >= NOMINAL:
            chosen, reached = g, True
    return chosen, reached, pd.DataFrame(
        curve, columns=["value", "cov_noshift", "cov_shift"])

## 8. Splits, calibration and bootstrap (do not edit)

In [8]:
# ------------------------------ ONE SPLIT ------------------------------
def split_stores(stores, split_id):
    rng = np.random.default_rng(SPLIT_SEED + split_id)
    perm = list(rng.permutation(stores))
    return set(perm[:N_CAL_STORES]), set(perm[N_CAL_STORES:])


def evaluate_split(tab, stores, split_id):
    cal_s, test_s = split_stores(stores, split_id)
    cal = tab[tab.treated.isin(cal_s) & tab.control.isin(cal_s)]
    test = tab[tab.treated.isin(test_s) & tab.control.isin(test_s)]

    lam, lam_ok, lam_curve = select_param(cal, LAMBDA_GRID, half_ba)
    k, k_ok, k_curve = select_param(cal, K_GRID, half_fixed)

    cal_ns = cal[cal.scenario == "noshift"].drop_duplicates("control")
    c_cal = conformal_threshold(np.abs(cal_ns["z"]))

    res = {"split": split_id, "lambda": lam, "k": k,
           "lambda_reached": lam_ok, "k_reached": k_ok,
           "c_cal": c_cal, "n_cal_pairs": len(cal) // 2,
           "n_test_pairs": len(test) // 2, "by_scenario": {},
           "lam_curve": lam_curve, "k_curve": k_curve,
           "cal": cal, "test": test, "test_stores": sorted(test_s)}

    for sc in SCENARIOS:
        d = test[test.scenario == sc]
        w_std = 2 * half_standard(d)
        w_fix = 2 * half_fixed(d, k)
        w_ba = 2 * half_ba(d, lam)
        k_eq = w_ba.mean() / w_std.mean()
        w_eq = 2 * half_fixed(d, k_eq)
        err = d["est"].to_numpy() - d["truth"].to_numpy()

        flag196 = np.abs(d["z"].to_numpy()) > 1.96
        flagcal = np.abs(d["z"].to_numpy()) > c_cal
        corrected = d["est"].to_numpy() - flagcal * d["corr_term"].to_numpy()
        err_c = corrected - d["truth"].to_numpy()

        res["by_scenario"][sc] = {
            "n_pairs": len(d),
            "n_distinct_controls": d["control"].nunique(),
            "cov_std": covered(d, half_standard(d)).mean(),
            "cov_fix": covered(d, half_fixed(d, k)).mean(),
            "cov_ba": covered(d, half_ba(d, lam)).mean(),
            "cov_fix_eq": covered(d, half_fixed(d, k_eq)).mean(),
            "w_std": w_std.mean(), "w_fix": w_fix.mean(),
            "w_ba": w_ba.mean(), "w_fix_eq": w_eq.mean(),
            "k_eq": k_eq,
            "mae": np.abs(err).mean(), "bias": err.mean(),
            "mae_corr": np.abs(err_c).mean(), "bias_corr": err_c.mean(),
            "flag196": flag196.mean(), "flagcal": flagcal.mean(),
            "n_flag196": int(d.loc[flag196, "control"].nunique()),
            "n_flagcal": int(d.loc[flagcal, "control"].nunique()),
            "pct_corrected": flagcal.mean(),
        }
    return res


# ------------------------------ BOOTSTRAP ------------------------------
def bootstrap_diff(res, sc, rng):
    """95% CI for coverage(BA) - coverage(fixed at BA width), resampling
    treated stores in the test set."""
    d = res["test"][res["test"].scenario == sc]
    k_eq = res["by_scenario"][sc]["k_eq"]
    d = d.assign(c_ba=covered(d, half_ba(d, res["lambda"])),
                 c_fx=covered(d, half_fixed(d, k_eq)))
    per = d.groupby("treated")[["c_ba", "c_fx"]].mean()
    arr = per.to_numpy()
    diffs = []
    for _ in range(N_BOOTSTRAP):
        idx = rng.integers(0, len(arr), len(arr))
        m = arr[idx].mean(axis=0)
        diffs.append(m[0] - m[1])
    return np.percentile(diffs, [2.5, 97.5]) * 100

## 9. Shift detection (do not edit)

In [9]:
# ------------------------------ DETECTOR EVALUATION ------------------------------
def _auc(no_shift_abs, shift_abs):
    """P(|z| of a shifted store > |z| of an unshifted store); 0.5 = chance."""
    a, b = np.asarray(no_shift_abs), np.asarray(shift_abs)
    return float(np.mean([(x > y) + 0.5 * (x == y) for y in a for x in b]))


def _control_z(tab, group, col, scenario):
    d = tab[(tab.scenario == scenario) & tab.control.isin(group)]
    return np.abs(d.drop_duplicates("control").set_index("control")[col])


def detector_comparison(tab, stores):
    """Both diagnostics x three threshold rules, over all random splits.
    Calibration uses calibration control stores only; every rate is on the
    test control stores (one value per store)."""
    rows = []
    for diag, col in (("yoy", "z_yoy"), ("legacy", "z_legacy")):
        for rule in ("fixed 1.96", "95% quantile (old rule)",
                     "conformal (n+1) rule"):
            fa, de, au, cs = [], [], [], []
            for sp in range(N_SPLITS):
                cal_s, test_s = split_stores(stores, sp)
                v = _control_z(tab, cal_s, col, "noshift").to_numpy()
                c = (1.96 if rule == "fixed 1.96" else
                     float(np.quantile(v, 0.95)) if rule.startswith("95%")
                     else conformal_threshold(v))
                zn = _control_z(tab, test_s, col, "noshift")
                zs = _control_z(tab, test_s, col, "shift")
                fa.append((zn > c).mean()); de.append((zs > c).mean())
                au.append(_auc(zn.to_numpy(), zs.to_numpy())); cs.append(c)
            rows.append({"diagnostic": diag, "threshold_rule": rule,
                         "mean_threshold": np.mean(cs),
                         "false_alarm_pct": 100 * np.mean(fa),
                         "false_alarm_sd": 100 * np.std(fa),
                         "detection_pct": 100 * np.mean(de),
                         "detection_sd": 100 * np.std(de),
                         "auc": np.mean(au)})
    return pd.DataFrame(rows)


def power_by_shift(wide, stores):
    """Detection rate vs size of the common shift (conformal threshold),
    averaged over the random splits."""
    z = {}
    for s_ in stores:
        o = wide[s_].to_numpy(float)
        z[(s_, 0.0)] = control_diagnostics(o)
        for sz in SHIFT_SIZES:
            x = o.copy(); x[PRE_WEEKS:] *= 1 + sz
            z[(s_, sz)] = control_diagnostics(x)
    rows = []
    for diag in ("yoy", "legacy"):
        for sz in SHIFT_SIZES:
            de = []
            for sp in range(N_SPLITS):
                cal_s, test_s = split_stores(stores, sp)
                c = conformal_threshold(
                    [abs(z[(q, 0.0)][diag]["z"]) for q in cal_s])
                de.append(np.mean([abs(z[(q, sz)][diag]["z"]) > c
                                   for q in test_s]))
            rows.append({"diagnostic": diag, "shift_pct": 100 * sz,
                         "detection_pct": 100 * np.mean(de)})
    return pd.DataFrame(rows)

## 10. Control-store baseline: BSTS with the control store as input (do not edit)

In [10]:
# ------------------------------ CONTROL-STORE BASELINE ------------------------------
def control_baseline_fits(wide, pairs):
    """Fit BSTS CausalImpact WITH the control store as a covariate for each
    (treated, control) pair and scenario. Progress is saved to disk, so an
    interrupted run resumes; failed fits are retried on the next run."""
    dates = wide.index
    cache_file = OUTPUT_DIR / "cache_bsts_control.json"
    cache = json.loads(cache_file.read_text()) if cache_file.exists() else {}
    key = lambda t, c, sc: f"{t}|{c}|{sc}"
    todo = sum(1 for t, c in pairs for sc in SCENARIOS
               if cache.get(key(t, c, sc)) is None)
    print(f"\nControl-store baseline: {len(pairs)} pairs x 2 scenarios; "
          f"{todo} fits still to run.")
    for i, (t, c) in enumerate(pairs, 1):
        o_t = wide[t].to_numpy(float); o_c = wide[c].to_numpy(float)
        for sc in SCENARIOS:
            if cache.get(key(t, c, sc)) is None:
                try:
                    cache[key(t, c, sc)] = list(bsts_interval(
                        make_series(o_t, sc, True), dates,
                        control=make_series(o_c, sc, False)))
                except Exception as exc:
                    cache[key(t, c, sc)] = None
                    print(f"  FIT FAILED {key(t, c, sc)}: "
                          f"{type(exc).__name__}: {exc}")
        if i % 10 == 0 or i == len(pairs):
            cache_file.write_text(json.dumps(cache))
            print(f"  control-store fits: pair {i}/{len(pairs)}")
    rows = []
    for t, c in pairs:
        for sc in SCENARIOS:
            v = cache.get(key(t, c, sc))
            if v is not None:
                rows.append({"treated": t, "control": c, "scenario": sc,
                             "est_c": v[0], "sigma_c": v[1]})
    return pd.DataFrame(rows)


def as_control_engine(d, ctrl):
    """Pairs in `d` that have a control-store fit, with est/sigma replaced by
    the control-store BSTS values, so the same helpers (covered, half_fixed,
    select_param) can be reused for the control-store method."""
    m = d.merge(ctrl, on=["treated", "control", "scenario"], how="inner")
    return m.assign(est=m["est_c"], sigma=m["sigma_c"])


def select_k_control(cal, ctrl):
    """Fixed widening k for the control-store baseline, chosen on the
    CALIBRATION pairs only (same rule as for the other methods)."""
    cf = as_control_engine(cal, ctrl)
    if cf.empty:
        return None, False, None
    return select_param(cf, K_GRID, half_fixed)


def control_metrics(test, ctrl, lam, k, k_ctrl=None):
    """Standard BSTS, BSTS + control store, fixed widening and break-aware,
    on the pairs in `test` for which a control-store fit exists. Works for
    test pairs and for calibration pairs alike."""
    d = test.merge(ctrl, on=["treated", "control", "scenario"], how="inner")
    res = {}
    for sc in SCENARIOS:
        s = d[d.scenario == sc]
        if s.empty:
            continue
        truth = s["truth"].to_numpy()
        half_c = Z95 * s["sigma_c"].to_numpy()
        err_c = s["est_c"].to_numpy() - truth
        cov_c = np.abs(err_c) <= half_c
        w_std = 2 * half_standard(s); w_c = 2 * half_c
        w_ba = 2 * half_ba(s, lam); w_fx = 2 * half_fixed(s, k)
        k_eq = w_ba.mean() / w_c.mean()          # control baseline at BA width
        cov_c_eq = np.abs(err_c) <= k_eq * half_c
        cov_std = covered(s, half_standard(s)); cov_ba = covered(s, half_ba(s, lam))
        cov_fx = covered(s, half_fixed(s, k))
        err_s = s["est"].to_numpy() - truth
        r = {
            "n_pairs": len(s), "n_treated": s["treated"].nunique(),
            "cov_std": cov_std.mean(), "cov_ctrl": cov_c.mean(),
            "cov_fix": cov_fx.mean(), "cov_ba": cov_ba.mean(),
            "cov_ctrl_at_ba_width": cov_c_eq.mean(), "k_eq_ctrl": k_eq,
            "w_std": w_std.mean(), "w_ctrl": w_c.mean(),
            "w_fix": w_fx.mean(), "w_ba": w_ba.mean(),
            "mae_std": np.abs(err_s).mean(), "mae_ctrl": np.abs(err_c).mean(),
            "bias_std": err_s.mean(), "bias_ctrl": err_c.mean()}
        frame = s.assign(c_ctrl=cov_c, c_ba=cov_ba,
                         ae_std=np.abs(err_s), ae_ctrl=np.abs(err_c))
        if k_ctrl is not None:
            cov_cc = np.abs(err_c) <= k_ctrl * half_c
            r["cov_ctrl_cal"] = cov_cc.mean()
            r["w_ctrl_cal"] = (2 * k_ctrl * half_c).mean()
            frame = frame.assign(c_ctrl_cal=cov_cc)
        r["_frame"] = frame
        res[sc] = r
    return res


def control_table(m, lam, k, k_ctrl=None):
    """One long table: the four methods (+ variants) per scenario."""
    rows = []
    for sc, label in (("noshift", "No business shift"), ("shift", "5% business shift")):
        if sc not in m:
            continue
        r = m[sc]
        specs = [
            ("Standard BSTS (treated store only)", r["cov_std"], r["w_std"], "-",
             r["mae_std"], r["bias_std"]),
            ("BSTS + control store (baseline)", r["cov_ctrl"], r["w_ctrl"], "-",
             r["mae_ctrl"], r["bias_ctrl"]),
            ("Fixed widening", r["cov_fix"], r["w_fix"], f"k={k}",
             r["mae_std"], r["bias_std"]),            # widening keeps the estimate
            ("Break-aware (proposed)", r["cov_ba"], r["w_ba"], f"lambda={lam}",
             r["mae_std"], r["bias_std"]),
            ("BSTS + control store at break-aware width", r["cov_ctrl_at_ba_width"],
             r["w_ba"], f"k={r['k_eq_ctrl']:.2f}", r["mae_ctrl"], r["bias_ctrl"])]
        if k_ctrl is not None:
            specs.append(("BSTS + control store, widened (k chosen on calibration)",
                          r["cov_ctrl_cal"], r["w_ctrl_cal"], f"k={k_ctrl}",
                          r["mae_ctrl"], r["bias_ctrl"]))
        for name, cov, w, par, mae, bias in specs:
            rows.append({"condition": label, "method": name, "parameter": par,
                         "pairs": r["n_pairs"], "treated_stores": r["n_treated"],
                         "coverage_pct": 100 * cov, "mean_width": w,
                         "width_vs_standard": w / r["w_std"],
                         "mae": mae, "bias": bias})
    return pd.DataFrame(rows)


def _boot_paired(frame, a, b, rng):
    """95% bootstrap CI of mean(a - b), resampling treated stores."""
    per = frame.groupby("treated")[[a, b]].mean().to_numpy()
    diffs = [(lambda m: m[0] - m[1])(per[rng.integers(0, len(per), len(per))]
                                     .mean(axis=0)) for _ in range(N_BOOTSTRAP)]
    return float(per[:, 0].mean() - per[:, 1].mean()), np.percentile(diffs, [2.5, 97.5])


def best_control_pairs(wide, group_pairs):
    """For every treated store, the control store (from the same group of
    stores, so the calibration/test separation is kept) with the highest
    Pearson correlation of weekly sales over the PRE_WEEKS pre-intervention
    weeks. Only pre-period data is used, so nothing from the post-period
    leaks into the choice."""
    corr = wide.iloc[:PRE_WEEKS].corr()
    rows = []
    for t in sorted(group_pairs["treated"].unique()):
        cands = sorted(set(group_pairs.loc[group_pairs.treated == t, "control"]))
        cs = corr.loc[t, cands]
        rows.append({"treated": int(t), "control": int(cs.idxmax()),
                     "pre_corr": float(cs.max())})
    return pd.DataFrame(rows), corr


def control_best_vs_random(test, ctrl, best, corr, lam, k):
    """BSTS with the control store: random control (every control store of
    the test group, i.e. all test pairs) versus the best-correlated control
    (one pair per treated store)."""
    rows, frames = [], {}
    pair_corr = test[test.scenario == "noshift"].drop_duplicates(
        ["treated", "control"])[["treated", "control"]]
    pair_corr = pair_corr.assign(r=[corr.loc[t, c] for t, c in
                                    zip(pair_corr.treated, pair_corr.control)])
    for name, sub in (("Random control (all test pairs)", None),
                      ("Best control (highest pre-period correlation)", best)):
        t_sub = test if sub is None else test.merge(
            sub[["treated", "control"]], on=["treated", "control"], how="inner")
        m = control_metrics(t_sub, ctrl, lam, k)
        r_mean = (pair_corr.r.mean() if sub is None else sub.pre_corr.mean())
        for sc, label in (("noshift", "No business shift"),
                          ("shift", "5% business shift")):
            if sc not in m:
                continue
            r = m[sc]
            frames[(name, sc)] = r["_frame"]
            rows.append({
                "condition": label, "control_choice": name,
                "pairs": r["n_pairs"], "mean_pre_corr": r_mean,
                "coverage_standard_pct": 100 * r["cov_std"],
                "coverage_ctrl_pct": 100 * r["cov_ctrl"],
                "coverage_breakaware_pct": 100 * r["cov_ba"],
                "width_standard": r["w_std"], "width_ctrl": r["w_ctrl"],
                "width_ctrl_vs_standard": r["w_ctrl"] / r["w_std"],
                "width_breakaware_vs_standard": r["w_ba"] / r["w_std"],
                "mae_standard": r["mae_std"], "mae_ctrl": r["mae_ctrl"],
                "bias_standard": r["bias_std"], "bias_ctrl": r["bias_ctrl"]})
    return pd.DataFrame(rows), frames


def pooled_equal_width(P, shares=(0.0, 0.1, 0.25, 0.5, 0.75, 1.0)):
    """Stricter equal-width test. One fixed multiplier used in BOTH scenarios
    (so it is not told whether a shift occurred), set so that its mean width
    equals the break-aware mean width in a population in which a share p of
    the pairs is shifted. Coverage is pooled over the two scenarios."""
    t, lam = P["test"], P["lambda"]
    d0, d1 = t[t.scenario == "noshift"], t[t.scenario == "shift"]
    cov = lambda d, h: float(np.mean(covered(d, h)))
    rows = []
    for p in shares:
        w_ba = (1 - p) * (2 * half_ba(d0, lam)).mean() + p * (2 * half_ba(d1, lam)).mean()
        w_st = (1 - p) * (2 * half_standard(d0)).mean() + p * (2 * half_standard(d1)).mean()
        kk = w_ba / w_st
        rows.append({
            "share_shifted": p, "lambda": lam, "width_vs_standard": kk,
            "coverage_standard_pct": 100 * ((1 - p) * cov(d0, half_standard(d0)) + p * cov(d1, half_standard(d1))),
            "coverage_breakaware_pct": 100 * ((1 - p) * cov(d0, half_ba(d0, lam)) + p * cov(d1, half_ba(d1, lam))),
            "coverage_single_fixed_pct": 100 * ((1 - p) * cov(d0, kk * half_standard(d0)) + p * cov(d1, kk * half_standard(d1)))})
    return pd.DataFrame(rows)

## 11. Main program: builds every table (do not edit)

In [11]:
# ------------------------------ MAIN ------------------------------
def main():
    wide = load_walmart()
    stores = list(wide.columns)
    print(f"ENGINE = {ENGINE} | stores={len(stores)} weeks={len(wide)} "
          f"pre={PRE_WEEKS} post={len(wide) - PRE_WEEKS}")
    tab = build_pair_table(wide)
    tab.to_csv(OUTPUT_DIR / "pair_level_inputs.csv", index=False)

    results = [evaluate_split(tab, stores, s) for s in range(N_SPLITS)]
    P = results[0]
    ns, sh = P["by_scenario"]["noshift"], P["by_scenario"]["shift"]
    lam, k, c_cal = P["lambda"], P["k"], P["c_cal"]
    out = {}

    print(f"\nPrimary split: {P['n_cal_pairs']} calibration pairs, "
          f"{P['n_test_pairs']} test pairs.")
    print(f"Selected on calibration: lambda={lam} (reached 95%: "
          f"{P['lambda_reached']}), k={k} (reached 95%: {P['k_reached']}), "
          f"calibrated threshold c={c_cal:.2f}")

    # ---- Tables 8.1 / 8.2 / 8.3 / 8.5 / 8.6 (one long table) ----
    rows = []
    for label, sc, r in (("No business shift", "noshift", ns),
                         ("5% business shift", "shift", sh)):
        for method, cov, w, par in (
                ("Standard", r["cov_std"], r["w_std"], "-"),
                (f"Fixed widening", r["cov_fix"], r["w_fix"], f"k={k}"),
                ("Break-aware", r["cov_ba"], r["w_ba"], f"lambda={lam}"),
                ("Fixed widening at break-aware width", r["cov_fix_eq"],
                 r["w_fix_eq"], f"k={r['k_eq']:.2f}")):
            rows.append({
                "condition": label, "method": method, "parameter": par,
                "test_pairs": r["n_pairs"], "coverage_pct": 100 * cov,
                "mean_width": w, "width_vs_standard": w / r["w_std"],
                "mae": r["mae"], "bias": r["bias"]})
    t_main = pd.DataFrame(rows)
    out["main"] = t_main
    print("\n=== TABLES 8.1, 8.2, 8.3, 8.5, 8.6: coverage, width, error "
          "(test pairs) ===")
    print(t_main.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

    rng = np.random.default_rng(SPLIT_SEED)
    ci = {sc: bootstrap_diff(P, sc, rng) for sc in SCENARIOS}
    print("\nBreak-aware minus fixed-at-equal-width coverage, 95% bootstrap "
          "CI over test treated stores (percentage points):")
    for sc in SCENARIOS:
        diff = 100 * (P["by_scenario"][sc]["cov_ba"]
                      - P["by_scenario"][sc]["cov_fix_eq"])
        print(f"  {sc}: difference {diff:+.2f}, CI [{ci[sc][0]:.2f}, "
              f"{ci[sc][1]:.2f}]")

    # ---- stricter equal-width test: one multiplier for both scenarios ----
    t_pool = pooled_equal_width(P)
    t_pool_sp = (pd.concat([pooled_equal_width(r_) for r_ in results])
                 .groupby("share_shifted").mean().reset_index())
    out["equal_width_pooled"] = t_pool
    out["equal_width_pooled_splits"] = t_pool_sp
    print("\n=== STRICTER EQUAL-WIDTH TEST: one fixed multiplier for both scenarios, "
          "same mean width as break-aware (primary split, test pairs) ===")
    print(t_pool.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
    print(f"\nSame, averaged over {N_SPLITS} random splits:")
    print(t_pool_sp.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

    # ---- Table 8.7: detection / false alarms ----
    t_det = detector_comparison(tab, stores)
    out["detector_comparison"] = t_det
    print("\n=== TABLE 8.7: detectors compared over "
          f"{N_SPLITS} random splits (test control stores; threshold "
          "calibrated on calibration stores) ===")
    print(t_det.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

    t_primary = pd.DataFrame([
        {"diagnostic": DIAGNOSTIC,
         "threshold": "c = 1.96",
         "false_alarm_rate_pct": 100 * ns["flag196"],
         "detection_rate_pct": 100 * sh["flag196"],
         "distinct_controls_flagged_noshift": ns["n_flag196"],
         "distinct_controls_flagged_shift": sh["n_flag196"],
         "distinct_test_controls": ns["n_distinct_controls"]},
        {"diagnostic": DIAGNOSTIC,
         "threshold": f"c = {c_cal:.2f} (conformal, 5% target)",
         "false_alarm_rate_pct": 100 * ns["flagcal"],
         "detection_rate_pct": 100 * sh["flagcal"],
         "distinct_controls_flagged_noshift": ns["n_flagcal"],
         "distinct_controls_flagged_shift": sh["n_flagcal"],
         "distinct_test_controls": ns["n_distinct_controls"]}])
    out["detection"] = t_primary
    print("\n=== TABLE 8.7 (primary split): active diagnostic "
          f"'{DIAGNOSTIC}' ===")
    print(t_primary.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

    t_power = power_by_shift(wide, stores)
    out["detector_power"] = t_power
    print("\n=== DETECTION RATE BY SHIFT SIZE (conformal threshold, "
          f"mean over {N_SPLITS} splits) ===")
    print(t_power.pivot(index="shift_pct", columns="diagnostic",
                        values="detection_pct").to_string(
        float_format=lambda v: f"{v:,.1f}"))

    # ---- Table 8.8: point-estimate correction ----
    t_corr = pd.DataFrame([
        {"condition": lab, "mae_standard": r["mae"],
         "mae_corrected": r["mae_corr"],
         "mae_change_pct": 100 * (r["mae_corr"] / r["mae"] - 1),
         "bias_standard": r["bias"], "bias_corrected": r["bias_corr"],
         "pct_pairs_corrected": 100 * r["pct_corrected"]}
        for lab, r in (("No business shift", ns), ("5% business shift", sh))])
    out["correction"] = t_corr
    print("\n=== TABLE 8.8: point-estimate correction (test pairs, "
          f"threshold c={c_cal:.2f}) ===")
    print(t_corr.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

    # ---- Table 8.4: Store 1 / Store 2 diagnostic example ----
    d = tab[(tab.treated == DEMO_TREATED) & (tab.control == DEMO_CONTROL)
            & (tab.scenario == "shift")].iloc[0]
    t_demo = pd.DataFrame({
        "metric": ["Legacy: control shift estimate (mean weekly residual)",
                   "Legacy: HAC standard error", "HAC lag",
                   "Legacy: control z-score",
                   "YoY: change in log year-over-year growth (S)",
                   "YoY: implied relative shift", "YoY: control z-score",
                   "lambda", "Additional interval margin (active diagnostic)",
                   "Standard cumulative estimate", "Standard CI width",
                   "Break-aware CI width (active diagnostic)",
                   "True cumulative effect"],
        "value": [d["delta"], d["hac_se"], HAC_LAG, d["z_legacy"],
                  d["yoy_S"], d["yoy_rel"], d["z_yoy"], LAMBDA_DEMO,
                  LAMBDA_DEMO * abs(d["z"]) * d["sigma"], d["est"],
                  2 * Z95 * d["sigma"],
                  2 * (Z95 + LAMBDA_DEMO * abs(d["z"])) * d["sigma"],
                  d["truth"]]})
    out["demo"] = t_demo
    print("\n=== TABLE 8.4: Store 1 (treated) / Store 2 (control), "
          "5% shift ===")
    print(t_demo.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

    # ---- Section 8.8: why the standard interval undercovers ----
    all_ns = tab[tab.scenario == "noshift"].drop_duplicates("treated")
    all_sh = tab[tab.scenario == "shift"].drop_duplicates("treated")
    err = (all_ns["est"] - all_ns["truth"]).to_numpy()
    sig = all_ns["sigma"].to_numpy()
    n_all = len(tab) // 2
    sec88 = {
        "coverage_all_pairs_noshift_pct": 100 * covered(
            tab[tab.scenario == "noshift"],
            half_standard(tab[tab.scenario == "noshift"])).mean(),
        "coverage_all_pairs_shift_pct": 100 * covered(
            tab[tab.scenario == "shift"],
            half_standard(tab[tab.scenario == "shift"])).mean(),
        "treated_stores_covered_noshift":
            f"{int(covered(all_ns, half_standard(all_ns)).sum())} of {len(all_ns)}",
        "treated_stores_covered_shift":
            f"{int(covered(all_sh, half_standard(all_sh)).sum())} of {len(all_sh)}",
        "sd_of_error_over_se": float(np.std(err / sig, ddof=1)),
        "mae_over_halfwidth": float(np.abs(err).mean() / (Z95 * sig).mean()),
        "share_estimate_above_truth_pct": 100 * float((err > 0).mean()),
        "all_pairs": n_all}
    out["sec88"] = pd.DataFrame([sec88])
    print("\n=== SECTION 8.8: standard interval, all pairs ===")
    for key, v in sec88.items():
        print(f"  {key}: {v:,.4f}" if isinstance(v, float) else f"  {key}: {v}")

    # ---- Stability over random splits ----
    srows = []
    for r in results:
        a, b = r["by_scenario"]["noshift"], r["by_scenario"]["shift"]
        srows.append({
            "split": r["split"], "lambda": r["lambda"], "k": r["k"],
            "c_cal": r["c_cal"],
            "cov_ba_noshift": 100 * a["cov_ba"], "cov_ba_shift": 100 * b["cov_ba"],
            "cov_fix_noshift": 100 * a["cov_fix"], "cov_fix_shift": 100 * b["cov_fix"],
            "ba_width_x_noshift": a["w_ba"] / a["w_std"],
            "ba_width_x_shift": b["w_ba"] / b["w_std"],
            "fix_width_x": a["w_fix"] / a["w_std"],
            "false_alarm_cal_pct": 100 * a["flagcal"],
            "detection_cal_pct": 100 * b["flagcal"]})
    t_split = pd.DataFrame(srows)
    out["splits"] = t_split
    print(f"\n=== STABILITY OVER {N_SPLITS} RANDOM SPLITS (test pairs) ===")
    print(t_split.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
    print("\nMeans over splits:")
    print(t_split.drop(columns="split").mean().to_string(
        float_format=lambda v: f"{v:,.2f}"))
    print(f"lambda range: {t_split['lambda'].min()} to {t_split['lambda'].max()}"
          f" | k range: {t_split['k'].min()} to {t_split['k'].max()}")

    # ---- Calibration curves (Fig 8.7) and coverage-vs-width (Fig 8.8) ----
    lam_curve = P["lam_curve"].assign(method="lambda")
    k_curve = P["k_curve"].assign(method="k")
    pd.concat([lam_curve, k_curve]).to_csv(
        OUTPUT_DIR / "fig8_7_calibration_curves.csv", index=False)

    wrows = []
    for sc in SCENARIOS:
        d = P["test"][P["test"].scenario == sc]
        w0 = (2 * half_standard(d)).mean()
        for g in LAMBDA_GRID:
            wrows.append({"scenario": sc, "method": "break-aware",
                          "param": g, "width_x": (2 * half_ba(d, g)).mean() / w0,
                          "coverage": covered(d, half_ba(d, g)).mean()})
        for g in K_GRID:
            wrows.append({"scenario": sc, "method": "fixed",
                          "param": g, "width_x": (2 * half_fixed(d, g)).mean() / w0,
                          "coverage": covered(d, half_fixed(d, g)).mean()})
    wdf = pd.DataFrame(wrows)
    wdf.to_csv(OUTPUT_DIR / "fig8_8_coverage_vs_width.csv", index=False)

    try:
        import matplotlib
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        for sc, ls in (("cov_noshift", "-"), ("cov_shift", "--")):
            ax[0].plot(P["lam_curve"]["value"].clip(lower=0.1),
                       P["lam_curve"][sc], ls, label=f"{sc[4:]}")
            ax[1].plot(P["k_curve"]["value"], P["k_curve"][sc], ls,
                       label=f"{sc[4:]}")
        ax[0].set_xscale("log")
        ax[0].axvline(max(lam, 0.1), ls=":", c="k")
        ax[1].axvline(k, ls=":", c="k")
        for a_, t_ in zip(ax, ("lambda (break-aware)", "k (fixed multiplier)")):
            a_.axhline(NOMINAL, c="grey", lw=0.8)
            a_.set_xlabel(t_); a_.set_ylabel("Calibration coverage"); a_.legend()
        fig.tight_layout(); fig.savefig(OUTPUT_DIR / "fig8_7.png", dpi=200)

        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        for a_, sc in zip(ax, SCENARIOS):
            for m, c_ in (("fixed", "C0"), ("break-aware", "C3")):
                s_ = wdf[(wdf.scenario == sc) & (wdf.method == m)]
                a_.plot(s_["width_x"], s_["coverage"], "o-", c=c_, label=m)
            s_ = P["by_scenario"][sc]
            a_.plot([1], [s_["cov_std"]], "ks", label="standard")
            a_.axhline(NOMINAL, c="grey", lw=0.8)
            a_.set_xscale("log"); a_.set_title(sc)
            a_.set_xlabel("Mean width relative to standard")
            a_.set_ylabel("Test coverage"); a_.legend()
        fig.tight_layout(); fig.savefig(OUTPUT_DIR / "fig8_8.png", dpi=200)
        plt.close("all")
    except Exception as exc:
        print("Figures skipped:", exc)

    # ---- Control-store baseline (BSTS with the control store as input) ----
    k_ctrl = None
    if CONTROL_BASELINE != "off":
        if ENGINE != "bsts":
            raise ValueError("The control-store baseline needs ENGINE = 'bsts'.")

        def _pairs_of(frame):
            f = frame[frame.scenario == "noshift"][["treated", "control"]]
            return sorted(set(map(tuple, f.itertuples(index=False, name=None))))
        test_pairs, cal_pairs = _pairs_of(P["test"]), _pairs_of(P["cal"])
        best_ctrl, pre_corr = best_control_pairs(wide, P["test"])
        best_ctrl.to_csv(OUTPUT_DIR / "best_control_pairs.csv", index=False)
        best_pairs = sorted(set(map(tuple, best_ctrl[["treated", "control"]]
                                    .itertuples(index=False, name=None))))
        if CONTROL_BASELINE == "all":
            pairs = _pairs_of(tab)
        elif CONTROL_BASELINE == "primary":
            pairs = sorted(set(cal_pairs) | set(test_pairs))
        else:
            pairs = test_pairs
        if MAX_CONTROL_PAIRS is not None and MAX_CONTROL_PAIRS < len(pairs):
            prng = np.random.default_rng(SPLIT_SEED)
            if CONTROL_BASELINE == "primary":     # half calibration, half test
                n_c = MAX_CONTROL_PAIRS // 2
                n_t = MAX_CONTROL_PAIRS - n_c
                pick = lambda lst, n: [lst[i] for i in sorted(
                    prng.choice(len(lst), min(n, len(lst)), replace=False))]
                pairs = sorted(set(pick(cal_pairs, n_c)) | set(pick(test_pairs, n_t)))
            else:
                sel = prng.choice(len(pairs), MAX_CONTROL_PAIRS, replace=False)
                pairs = [pairs[i] for i in sorted(sel)]
            print(f"PILOT: only {len(pairs)} randomly chosen pairs are used.")
        pairs = sorted(set(pairs) | set(best_pairs))   # the best controls are always fitted
        print(f"Control-store baseline mode '{CONTROL_BASELINE}': {len(pairs)} pairs "
              f"({len(set(pairs) & set(cal_pairs))} calibration, "
              f"{len(set(pairs) & set(test_pairs))} test) x 2 scenarios.")
        ctrl = control_baseline_fits(wide, pairs)
        ctrl.to_csv(OUTPUT_DIR / "control_baseline_fits.csv", index=False)
        n_missing = 2 * len(pairs) - len(ctrl)
        if n_missing:
            print(f"WARNING: {n_missing} control-store fits failed and are left "
                  "out; run the cell again to retry them.")

        if CONTROL_BASELINE in ("primary", "all"):
            k_ctrl, k_ctrl_ok, k_ctrl_curve = select_k_control(P["cal"], ctrl)
            n_cal_fit = len(as_control_engine(P["cal"], ctrl)) // 2
            print(f"Control-store calibration: k_ctrl={k_ctrl} (reached 95%: "
                  f"{k_ctrl_ok}) on {n_cal_fit} calibration pairs")
            if k_ctrl_curve is not None:
                k_ctrl_curve.to_csv(OUTPUT_DIR / "control_baseline_k_curve.csv",
                                    index=False)

        m = control_metrics(P["test"], ctrl, lam, k, k_ctrl)
        t_ctrl = control_table(m, lam, k, k_ctrl)
        out["control_baseline"] = t_ctrl
        print("\n=== CONTROL-STORE BASELINE vs PROPOSED METHOD "
              f"(primary split, TEST pairs, {CONTROL_BASELINE} mode) ===")
        print(t_ctrl.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

        if CONTROL_BASELINE in ("primary", "all"):
            m_cal = control_metrics(P["cal"], ctrl, lam, k, k_ctrl)
            t_ctrl_cal = control_table(m_cal, lam, k, k_ctrl)
            out["control_baseline_calibration"] = t_ctrl_cal
            print("\n=== CONTROL-STORE BASELINE on the CALIBRATION pairs "
                  "(primary split) ===")
            print(t_ctrl_cal.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

        brng = np.random.default_rng(SPLIT_SEED + 1)
        print("\nPaired 95% bootstrap CIs over treated test stores:")
        for sc in SCENARIOS:
            if sc not in m: continue
            f = m[sc]["_frame"]
            d1, ci1 = _boot_paired(f, "c_ctrl", "c_ba", brng)
            d2, ci2 = _boot_paired(f, "ae_ctrl", "ae_std", brng)
            print(f"  {sc}: coverage(control baseline) - coverage(break-aware) = "
                  f"{100*d1:+.2f} pp, CI [{100*ci1[0]:.2f}, {100*ci1[1]:.2f}]")
            print(f"  {sc}: MAE(control baseline) - MAE(standard) = "
                  f"{d2:+,.0f}, CI [{ci2[0]:,.0f}, {ci2[1]:,.0f}]")
            if k_ctrl is not None:
                d3, ci3 = _boot_paired(f, "c_ctrl_cal", "c_ba", brng)
                print(f"  {sc}: coverage(control baseline, calibrated k) - "
                      f"coverage(break-aware) = {100*d3:+.2f} pp, "
                      f"CI [{100*ci3[0]:.2f}, {100*ci3[1]:.2f}]")

        t_bvr, frames_bvr = control_best_vs_random(P["test"], ctrl, best_ctrl,
                                                   pre_corr, lam, k)
        out["control_best_vs_random"] = t_bvr
        print("\n=== BSTS WITH CONTROL: RANDOM CONTROL vs BEST CONTROL "
              "(primary split, test pairs) ===")
        print("Best control = the control store with the highest correlation of "
              f"weekly sales over the {PRE_WEEKS} pre-intervention weeks.")
        print(t_bvr.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
        print("\nBest minus random control, paired over treated test stores "
              "(95% bootstrap CI):")
        for sc in SCENARIOS:
            fr = frames_bvr.get(("Random control (all test pairs)", sc))
            fb = frames_bvr.get(("Best control (highest pre-period correlation)", sc))
            if fr is None or fb is None:
                continue
            for col, lab, mult in (("c_ctrl", "coverage (pp)", 100.0),
                                   ("ae_ctrl", "absolute error", 1.0)):
                rr = fr.groupby("treated")[col].mean()
                bb = fb.groupby("treated")[col].mean()
                dd = (bb - rr.reindex(bb.index)).dropna().to_numpy()
                rng_b = np.random.default_rng(SPLIT_SEED + 2)
                boots = [dd[rng_b.integers(0, len(dd), len(dd))].mean()
                         for _ in range(N_BOOTSTRAP)]
                lo, hi = np.percentile(boots, [2.5, 97.5])
                print(f"  {sc}: {lab}: {mult*dd.mean():+,.2f}, "
                      f"CI [{mult*lo:,.2f}, {mult*hi:,.2f}] ({len(dd)} stores)")

        if CONTROL_BASELINE == "all":
            srows = []
            for r_ in results:
                kc, _, _ = select_k_control(r_["cal"], ctrl)
                mm = control_metrics(r_["test"], ctrl, r_["lambda"], r_["k"], kc)
                row = {"split": r_["split"], "lambda": r_["lambda"], "k": r_["k"],
                       "k_ctrl": kc}
                for sc in SCENARIOS:
                    if sc in mm:
                        for key_ in ("cov_ctrl", "cov_ba", "cov_fix",
                                     "cov_ctrl_at_ba_width", "cov_ctrl_cal"):
                            if key_ in mm[sc]:
                                row[f"{key_}_{sc}"] = 100 * mm[sc][key_]
                        row[f"ctrl_width_x_{sc}"] = mm[sc]["w_ctrl"] / mm[sc]["w_std"]
                        row[f"ba_width_x_{sc}"] = mm[sc]["w_ba"] / mm[sc]["w_std"]
                        row[f"mae_ctrl_{sc}"] = mm[sc]["mae_ctrl"]
                        row[f"mae_std_{sc}"] = mm[sc]["mae_std"]
                srows.append(row)
            t_cs = pd.DataFrame(srows)
            out["control_baseline_splits"] = t_cs
            print(f"\n=== CONTROL-STORE BASELINE OVER {N_SPLITS} RANDOM SPLITS (test pairs) ===")
            print(t_cs.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
            print("\nMeans over splits:")
            print(t_cs.drop(columns="split").mean().to_string(float_format=lambda v: f"{v:,.2f}"))

    # ---- Save everything ----
    for name, frame in out.items():
        frame.to_csv(OUTPUT_DIR / f"table_{name}.csv", index=False)
    (OUTPUT_DIR / "run_config.json").write_text(json.dumps({
        "engine": ENGINE, "pre_weeks": PRE_WEEKS,
        "treatment_effect": TREATMENT_EFFECT, "business_shift": BUSINESS_SHIFT,
        "shift_composition": SHIFT_COMPOSITION, "diagnostic": DIAGNOSTIC,
        "control_baseline": CONTROL_BASELINE,
        "threshold_rule": "conformal (n+1) on distinct calibration controls",
        "hac_lag": HAC_LAG, "n_cal_stores": N_CAL_STORES,
        "n_splits": N_SPLITS, "split_seed": SPLIT_SEED,
        "lambda_grid": LAMBDA_GRID, "k_grid": K_GRID,
        "selected_lambda": lam, "selected_k": k, "calibrated_c": c_cal,
        "selected_k_control_store": k_ctrl,
        "bootstrap_ci_pp": {sc: list(map(float, ci[sc])) for sc in SCENARIOS},
    }, indent=2))
    print(f"\nAll tables, figures and the run configuration were saved "
          f"to {OUTPUT_DIR.resolve()}")
    return out

## 12. Settings and run (the first output line must say `ENGINE = bsts`)
`pilot` = 20 pairs (10 calibration + 10 test; a quick test). `full` = the 462 calibration pairs + the 506 test pairs of the primary split, no-shift and 5%-shift (1,936 control-store fits plus 90 treated-store fits; the main comparison). `all` = all 1,980 pairs (adds the 20-split summary; very slow).

In [20]:




import numpy as np
import pandas as pd

def extract_causalimpact_results(impact):
    """
    Extract the cumulative causal effect estimate and confidence
    interval from different CausalImpact package result formats.
    """

    # Try summary_data when the installed version provides it.
    summary_data = getattr(impact, "summary_data", None)

    if isinstance(summary_data, pd.DataFrame):
        result = summary_data.copy()

    elif isinstance(summary_data, pd.Series):
        result = summary_data.to_frame().T

    else:
        result = None

    # Fall back to the summary DataFrame when available.
    if result is None:
        summary = getattr(impact, "summary", None)

        if isinstance(summary, pd.DataFrame):
            result = summary.copy()

        elif isinstance(summary, pd.Series):
            result = summary.to_frame().T

    if result is None:
        raise AttributeError(
            "Could not find a result DataFrame in the CausalImpact "
            "object. Inspect impact attributes and the installed "
            "package version."
        )

    # Normalize row and column labels.
    result.index = result.index.map(
        lambda x: str(x).strip().lower()
    )
    result.columns = result.columns.map(
        lambda x: str(x).strip().lower()
    )

    # Find the cumulative-effect row or column.
    cumulative_names = {"cumulative", "cum"}

    if any(str(x).lower() in cumulative_names for x in result.index):
        row = result.loc[
            result.index.isin(cumulative_names)
        ].iloc[0]

    elif any(str(x).lower() in cumulative_names for x in result.columns):
        row = result.loc[
            :, result.columns.isin(cumulative_names)
        ].iloc[:, 0]

    else:
        raise ValueError(
            "No cumulative-effect results found. Available rows: "
            f"{list(result.index)}; columns: {list(result.columns)}"
        )

    # Normalize keys for flexible column matching.
    values = {
        str(k).lower().replace("_", " ").strip(): v
        for k, v in row.items()
    }

    def get_value(*names):
        for name in names:
            if name in values:
                return float(values[name])
        return None

    estimate = get_value("abs effect", "absolute effect")
    lower = get_value("95% ci lower", "lower", "lower ci")
    upper = get_value("95% ci upper", "upper", "upper ci")

    if estimate is None or lower is None or upper is None:
        raise ValueError(
            "Could not extract the cumulative effect and confidence "
            f"interval. Available fields: {list(values.keys())}"
        )

    return estimate, lower, upper



## 13. Show the results

In [21]:
import pandas as pd, os
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
def show(path, title):
    print(title, "\n")
    print(pd.read_csv(path).round(2).to_string(index=False) if os.path.exists(path)
          else f"Table not found: {path} (the run cell did not finish).")
    print()
show("paper_tables/table_control_baseline.csv",
     "Control-store baseline vs proposed method (primary split, test pairs):")
show("paper_tables/table_control_baseline_calibration.csv",
     "Same methods on the calibration pairs (where the widening k is chosen):")
show("paper_tables/table_equal_width_pooled.csv",
     "Stricter equal-width test (one fixed multiplier for both scenarios, primary split):")
show("paper_tables/table_control_best_vs_random.csv",
     "BSTS with control: random control vs best control (highest pre-period correlation), test pairs:")
if os.path.exists("run_log.txt"):
    _lines = open("run_log.txt", encoding="utf-8").read().splitlines()
    for _i in range(len(_lines) - 1, -1, -1):          # last run only
        if _lines[_i].startswith("Control-store calibration"):
            print(_lines[_i]); break
    for _i in range(len(_lines) - 1, -1, -1):
        if _lines[_i].startswith("Paired 95%"):
            print("\nPaired bootstrap intervals:"); print("\n".join(_lines[_i + 1:_i + 7])); break
if os.path.exists("paper_tables/table_control_baseline_splits.csv"):
    print("\n20-split summary (means):")
    print(pd.read_csv("paper_tables/table_control_baseline_splits.csv").drop(columns="split").mean().round(2).to_string())
print("\nAll tables are saved in the folder:", os.path.abspath("paper_tables"))

Control-store baseline vs proposed method (primary split, test pairs): 

Table not found: paper_tables/table_control_baseline.csv (the run cell did not finish).

Same methods on the calibration pairs (where the widening k is chosen): 

Table not found: paper_tables/table_control_baseline_calibration.csv (the run cell did not finish).

Stricter equal-width test (one fixed multiplier for both scenarios, primary split): 

Table not found: paper_tables/table_equal_width_pooled.csv (the run cell did not finish).

BSTS with control: random control vs best control (highest pre-period correlation), test pairs: 

Table not found: paper_tables/table_control_best_vs_random.csv (the run cell did not finish).


All tables are saved in the folder: c:\Users\user\AppData\Local\Programs\Microsoft VS Code\paper_tables
